## Profiling sequencial — trechos paralelizáveis vs. sequenciais

Lê `results/profile_seq.txt` (saída do `sample` do macOS sobre o binário sequencial) e extrai a tabela **"Sort by top of stack, same collapsed"** — contagem de amostras por função, no topo da pilha. Classifica cada função como parte do laço paralelizável (`lenet_forward_backward` e suas chamadas: conv/dense/relu/maxpool forward+backward) ou sequencial (`lenet_sgd_update`, o update de pesos), e estima a fração sequencial — o teto de Amdahl medido por profiling, complementar à instrumentação por eixo (`tempo_parallel_region`/`tempo_reduction`) que `train.cpp` já imprime em tempo de execução.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

RESULTS_DIR = Path("../results")

In [ ]:
import re as _re

PROFILE_PATH = RESULTS_DIR / "profile_seq.txt"
profile_text = PROFILE_PATH.read_text()

section = profile_text.split("Sort by top of stack, same collapsed")[1].split("Binary Images:")[0]

profile_rows = []
for line in section.splitlines():
    m = _re.match(r"\s*(.+?)\s+\(in [\w.]+\)\s+(\d+)\s*$", line)
    if m:
        func, count = m.group(1).strip(), int(m.group(2))
        profile_rows.append((func, count))

print(f"{len(profile_rows)} funções encontradas")
for func, count in profile_rows:
    print(f"  {count:6d}  {func}")

### Classificar: paralelizável (laço de amostras) vs. sequencial

- **Paralelizável** — chamadas dentro de `lenet_forward_backward` (`train.cpp`, o `#pragma omp for`): as camadas da rede (conv/dense/relu/maxpool, forward e backward), `softmax_cross_entropy`, `argmax`.
- **Sequencial** — `lenet_sgd_update` (o update de pesos, fora da região paralela).
- **Runtime/alocação** — malloc/free/memset/memmove/`std::vector` etc. Ocorre tanto dentro quanto fora do laço (ex. `_M_fill_assign` ao zerar buffers), então é reportado à parte em vez de forçado numa das duas categorias.

In [ ]:
PARALLEL_PREFIXES = (
    "conv_forward", "conv_backward",
    "dense_forward", "dense_backward",
    "relu_forward", "relu_backward",
    "maxpool2x2_forward", "maxpool2x2_backward",
    "softmax_cross_entropy", "argmax",
    "lenet_forward_backward",
)
SEQUENTIAL_PREFIXES = ("lenet_sgd_update",)


def classify(func):
    if func.startswith(SEQUENTIAL_PREFIXES):
        return "sequencial (SGD update)"
    if func.startswith(PARALLEL_PREFIXES):
        return "paralelizável (forward/backward)"
    return "runtime/alocação"


profile_totals = {"paralelizável (forward/backward)": 0, "sequencial (SGD update)": 0, "runtime/alocação": 0}
for func, count in profile_rows:
    profile_totals[classify(func)] += count

profile_grand_total = sum(profile_totals.values())
for label, count in profile_totals.items():
    print(f"{label:35s} {count:6d} amostras  ({count/profile_grand_total:.2%})")

In [ ]:
from matplotlib.patches import Patch

def short_name(func):
    return func.split("(")[0]

sorted_rows = sorted(profile_rows, key=lambda r: r[1], reverse=True)
labels = [short_name(f) for f, _ in sorted_rows]
counts = [c for _, c in sorted_rows]
bar_colors = [
    "tab:blue" if classify(f) == "paralelizável (forward/backward)"
    else "tab:red" if classify(f) == "sequencial (SGD update)"
    else "tab:gray"
    for f, _ in sorted_rows
]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(labels, counts, color=bar_colors)
ax.invert_yaxis()
ax.set_xlabel("amostras (top of stack)")
ax.set_title("profile_seq.txt — amostras por função")
ax.legend(handles=[
    Patch(color="tab:blue", label="paralelizável (forward/backward)"),
    Patch(color="tab:red", label="sequencial (SGD update)"),
    Patch(color="tab:gray", label="runtime/alocação"),
], loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
labels2 = list(profile_totals.keys())
values2 = list(profile_totals.values())
colors2 = ["tab:blue", "tab:red", "tab:gray"]

fig, (ax_bar, ax_pie) = plt.subplots(1, 2, figsize=(11, 4))

ax_bar.bar(labels2, values2, color=colors2)
ax_bar.set_ylabel("amostras")
ax_bar.tick_params(axis="x", rotation=15)
for i, v in enumerate(values2):
    ax_bar.text(i, v, f"{v}\n({v/profile_grand_total:.1%})", ha="center", va="bottom")

ax_pie.pie(values2, labels=labels2, colors=colors2, autopct="%1.1f%%", startangle=90)
ax_pie.set_title("fração do tempo de CPU (por amostragem)")

plt.tight_layout()
plt.show()

seq_frac = profile_totals["sequencial (SGD update)"] / profile_grand_total
print(f"fração estritamente sequencial (SGD update): {seq_frac:.4%}")
print(f"fração paralelizável (forward/backward): {profile_totals['paralelizável (forward/backward)']/profile_grand_total:.4%}")